# PC-FNO3D réel NATIONAL (Tunisie, nœuds 1 km) — génération du jeu (Colab, CPU)

Vérité d'entraînement = le générateur forcé par **ERA5**, sur le relief (Copernicus DEM) et le sol
(ESA WorldCover) réels de toute la Tunisie ; **aucune station n'y entre**. Surface 500 m
(1656 × 776), nœuds 1 km, niveaux 25,50,75,100,150,200,300,400,500,650,800,1000,1250,1500,2000,2500 m au-dessus de la mer. 1 000 instants, mêmes blocs de
test et de validation que le Grand Tunis.

**Environnement : CPU, mémoire RAM élevée.** Un seul **Tout exécuter** par partie : le carnet fait
la partie suivante qui manque sur le Drive (`MyDrive/pcfno3d_national/`). **Place nécessaire : ~50 Go
sur le Drive** (affichée par la première cellule).

| partie | contenu | point de reprise |
|---|---|---|
| 1 | entraînement, 1re moitié | `cache_partie_1.zip` |
| 2 | entraînement, 2e moitié + validation | `cache_partie_2.zip` |
| 3 | test (janv.–mars et juillet–août 2022) | `cache_partie_3.zip` |

## 1. Drive, code et données

In [ ]:
from google.colab import drive
import os as _os
if not _os.path.isdir("/content/drive/MyDrive"):   # deja monte (panneau Fichiers) : ne pas redemander
    drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")   # code et poids du Grand Tunis
NAT = Path("/content/drive/MyDrive/pcfno3d_national")
NAT.mkdir(parents=True, exist_ok=True)
JOURNAUX = NAT / "journaux"
JOURNAUX.mkdir(exist_ok=True)
libre = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print(f"place libre sur le Drive : {libre:.0f} Go (le jeu national en demande ~50)")

CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille nationale, grille_reelle.json, forcage ERA5 (une archive sur le Drive)
DON = Path("/content/donnees")
G = DON / "national"
if not (G / "forcage_era5.csv").exists():
    with zipfile.ZipFile(NAT / "national_donnees.zip") as zf:
        zf.extractall(DON)
print(sorted(p.name for p in G.iterdir()))

## 2. Calendrier et partie à faire

In [ ]:
TEST = "2022-01-01:2022-03-31,2022-07-01:2022-08-31"     # memes blocs que le Grand Tunis
VAL = "2021-01-01:2021-02-28,2021-07-01:2021-07-31"
LOC = Path("/content/national")
JEU = LOC / "jeu"
JEU.mkdir(parents=True, exist_ok=True)
INFO = json.loads((G / "grille_reelle.json").read_text(encoding="utf-8"))
print("domaine :", INFO["zone"], "| echelle est-ouest :", INFO.get("facteur_echelle_est_ouest"))

if (NAT / "calendrier.json").exists():
    shutil.copy2(NAT / "calendrier.json", JEU / "calendrier.json")

PARTIES = {1: [("train", "1/2")], 2: [("train", "2/2"), ("val", None)], 3: [("test", None)]}
faites = [k for k in PARTIES if (NAT / f"cache_partie_{k}.zip").exists()]
PARTIE = next((k for k in PARTIES if k not in faites), None)
print("parties deja sur le Drive :", faites, "| a faire maintenant :", PARTIE)

## 3. Génération de la partie, puis point de reprise sur le Drive

In [ ]:
if PARTIE is None:
    print("les 3 parties sont sur le Drive : passer au carnet d'entrainement")
else:
    cache = LOC / f"cache_p{PARTIE}"
    if cache.exists():
        shutil.rmtree(cache)                 # partie interrompue : on la refait entiere
    nw = max(1, (os.cpu_count() or 2) - 1)
    base = (f"python -u scripts/generate_aneo_3d.py --input-grid {G / 'aneo_grid_points_surface.csv'} "
            f"--output-dir {JEU} --cache-dir {cache} --forcage {G / 'forcage_era5.csv'} "
            f"--nx {INFO['zone']['nx']} --ny {INFO['zone']['ny']} --dx {INFO['zone']['dx_m']} "
            f"--niveaux 25,50,75,100,150,200,300,400,500,650,800,1000,1250,1500,2000,2500 --f-coriolis {INFO['f_coriolis']} "
            f"--perturbations 0.4 --test {TEST} --val {VAL} --marge-jours 7 "
            f"--instants 1000 --heures-par-jour 4 --workers {nw}")
    for jeux, part in PARTIES[PARTIE]:
        lancer(f"generation_p{PARTIE}", base + f" --jeux {jeux}" + (f" --partie {part}" if part else ""))
    if not (NAT / "calendrier.json").exists():
        shutil.copy2(JEU / "calendrier.json", NAT / "calendrier.json")

    # jours du calendrier sans forcage pour cette partie : refuse le point de reprise
    cal = json.loads((JEU / "calendrier.json").read_text())
    import pandas as pd
    t = pd.to_datetime(pd.read_csv(G / "forcage_era5.csv", usecols=["timestamp_utc"])["timestamp_utc"], utc=True)
    forces = set(t.dt.tz_localize(None).dt.normalize())
    idx = pd.read_csv(cache / "index.csv")
    attendus = {k for jeux, _ in PARTIES[PARTIE] for k in cal["split"][f"{jeux}_days"]}
    dates = {d["day_index"]: d["date"] for d in cal["days"]}
    manque = sorted(dates[k] for k in attendus if pd.Timestamp(dates[k]) not in forces)
    print(f"partie {PARTIE} : {len(idx)} instants generes ; jours sans forcage ERA5 : {len(manque)}")
    if manque and PARTIE in (2, 3):
        raise RuntimeError(f"forcage ERA5 absent pour {manque[:5]}... : deposer un forcage a jour")

    tmp = NAT / f"cache_partie_{PARTIE}.zip.part"
    with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_STORED) as zf:
        for f in sorted(cache.iterdir()):
            nom = f"index_partie_{PARTIE}.csv" if f.name == "index.csv" else f.name
            zf.write(f, nom)
    tmp.replace(NAT / f"cache_partie_{PARTIE}.zip")
    if not (NAT / "jeu.zip").exists():
        with zipfile.ZipFile(NAT / "jeu.zip", "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
            for f in sorted(JEU.iterdir()):
                if f.is_file():
                    zf.write(f, f.name)
    print(f"point de reprise {PARTIE}/3 sur le Drive : "
          f"{(NAT / f'cache_partie_{PARTIE}.zip').stat().st_size / 1e9:.1f} Go")